# ☕ Random Forests: Coffee Quality Prediction in East Africa
**Author:** Jok Akech Atem Mabior | Carnegie Mellon University Africa  
**Dataset:** Synthetic Ethiopian & Rwandan Coffee Farm Data  
**Objective:** Predict coffee quality score from agronomic and environmental features

## Background

### East African Coffee Industry

Ethiopia is the **birthplace of coffee** — legend holds that a goat herder named Kaldi first noticed the energizing effect of coffee berries in the Kaffa region around 850 AD. Today, Ethiopia is Africa's largest coffee producer and the world's fifth largest, exporting over 250,000 metric tonnes annually. Its highland regions — Yirgacheffe, Sidama, Harrar, and Kaffa — produce some of the world's most celebrated specialty coffees, prized for their floral, fruity, and wine-like profiles.

**Rwanda** has transformed its coffee sector since the early 2000s as part of post-genocide economic recovery. The government invested heavily in washing stations and farmer cooperatives, and Rwandan coffee now commands premium prices on international specialty markets. The country's high altitude (1,500–2,000+ m), volcanic soils, and two rainfall seasons create ideal growing conditions.

### Coffee Quality Grading

The **Specialty Coffee Association (SCA)** uses a 100-point cupping protocol to evaluate coffee quality across attributes including:
- **Aroma, Flavor, Aftertaste, Acidity, Body, Balance, Uniformity, Clean Cup, Sweetness, Overall**

Scores are interpreted as:
| Score Range | Grade |
|-------------|-------|
| 90–100 | Exceptional |
| 85–89 | Excellent |
| 80–84 | Very Good (Specialty) |
| 70–79 | Good |
| 60–69 | Commercial |

### Why ML for Coffee Quality?

Traditional quality assessment requires trained Q-graders (SCA-certified) who are expensive and scarce in rural East Africa. Machine learning models trained on agronomic data (altitude, processing method, rainfall, etc.) can help **small-scale farmers predict quality before harvest**, optimize processing decisions, and earn higher prices. This has direct implications for rural livelihoods — coffee farming supports over 15 million Ethiopians alone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
print("Libraries loaded successfully!")

## 1. Data Generation

We generate a synthetic dataset of 800 coffee farm records from Ethiopia and Rwanda, incorporating realistic agronomic, environmental, and processing features that influence cup quality.

In [ ]:
np.random.seed(42)
n = 800

regions = ['Yirgacheffe', 'Sidama', 'Harrar', 'Kefa', 'Kigali', 'Eastern Province',
           'Southern Province', 'Western Province']
countries = ['Ethiopia', 'Ethiopia', 'Ethiopia', 'Ethiopia', 'Rwanda', 'Rwanda', 'Rwanda', 'Rwanda']
processing = ['Washed', 'Natural', 'Honey', 'Semi-washed']
varieties = ['Heirloom', 'Bourbon', 'Typica', 'SL28', 'Pacamara']

region_idx = np.random.randint(0, len(regions), n)
region = np.array(regions)[region_idx]
country = np.array(countries)[region_idx]
processing_method = np.random.choice(processing, n, p=[0.45, 0.3, 0.15, 0.1])
variety = np.random.choice(varieties, n)
harvest_month = np.random.randint(1, 13, n)

altitude_m = np.where(country == 'Ethiopia',
    np.random.normal(1900, 400, n).clip(1200, 3000),
    np.random.normal(1700, 300, n).clip(1300, 2500))

annual_rainfall_mm = np.random.normal(1400, 250, n).clip(800, 2200)
avg_temperature_c = 20 - 0.006 * altitude_m + np.random.normal(0, 1.5, n)
avg_temperature_c = avg_temperature_c.clip(14, 26)
shade_percentage = np.random.beta(3, 2, n) * 100
farm_size_ha = np.random.exponential(2, n).clip(0.25, 20)
organic_certified = np.random.binomial(1, 0.35, n)
fermentation_hours = np.where(processing_method == 'Washed',
    np.random.normal(36, 8, n).clip(18, 72),
    np.zeros(n))
cherry_ripeness = np.random.normal(85, 10, n).clip(60, 100)
years_experience = np.random.randint(1, 31, n)

# Quality score formula (SCA scale 60-95)
quality_score = (
    60 +
    0.006 * altitude_m +
    0.003 * annual_rainfall_mm -
    0.4 * abs(avg_temperature_c - 19) +
    0.05 * shade_percentage +
    3.0 * organic_certified +
    0.12 * cherry_ripeness +
    0.05 * years_experience +
    np.where(processing_method == 'Natural', 2.0, 0) +
    np.where(processing_method == 'Honey', 1.5, 0) +
    np.random.normal(0, 2.5, n)
).clip(60, 94)

df = pd.DataFrame({
    'region': region, 'country': country,
    'processing_method': processing_method, 'variety': variety,
    'harvest_month': harvest_month, 'altitude_m': altitude_m.round(0).astype(int),
    'annual_rainfall_mm': annual_rainfall_mm.round(1),
    'avg_temperature_c': avg_temperature_c.round(2),
    'shade_percentage': shade_percentage.round(1),
    'farm_size_ha': farm_size_ha.round(2),
    'organic_certified': organic_certified,
    'fermentation_hours': fermentation_hours.round(1),
    'cherry_ripeness': cherry_ripeness.round(1),
    'years_experience': years_experience,
    'quality_score': quality_score.round(2)
})

# Add quality grade categorical
df['quality_grade'] = pd.cut(df['quality_score'],
    bins=[60, 70, 80, 85, 90, 100],
    labels=['Commercial', 'Good', 'Very Good', 'Excellent', 'Exceptional'])

print(f"Dataset: {df.shape}")
print(f"Quality score stats:\n{df['quality_score'].describe().round(2)}")
print(f"\nGrade distribution:\n{df['quality_grade'].value_counts()}")
df.head()

## 2. Exploratory Data Analysis

Before building models, we explore the dataset to understand quality distributions, regional patterns, and relationships between features and coffee quality.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Quality score distribution
axes[0,0].hist(df['quality_score'], bins=30, color='#8B4513', edgecolor='white', alpha=0.8)
axes[0,0].set_xlabel('Quality Score (SCA)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Coffee Quality Score Distribution', fontweight='bold')
axes[0,0].axvline(df['quality_score'].mean(), color='red', linestyle='--',
                   label=f'Mean={df["quality_score"].mean():.1f}')
axes[0,0].legend()

# Grade distribution
grade_counts = df['quality_grade'].value_counts()
axes[0,1].pie(grade_counts.values, labels=grade_counts.index, autopct='%1.1f%%',
              colors=['#d4a017', '#c8a45a', '#6d9e24', '#2e7d32', '#1a5e1a'])
axes[0,1].set_title('Quality Grade Distribution', fontweight='bold')

# Quality by country
df.boxplot(column='quality_score', by='country', ax=axes[0,2])
axes[0,2].set_title('Quality Score by Country', fontweight='bold')
axes[0,2].set_xlabel('Country')
axes[0,2].set_ylabel('Quality Score')
plt.sca(axes[0,2]); plt.suptitle('')

# Quality by processing method
df.boxplot(column='quality_score', by='processing_method', ax=axes[1,0])
axes[1,0].set_title('Quality Score by Processing Method', fontweight='bold')
axes[1,0].set_xlabel('Processing Method')
plt.sca(axes[1,0]); plt.suptitle('')

# Altitude vs quality
sc = axes[1,1].scatter(df['altitude_m'], df['quality_score'],
                        c=df['organic_certified'], cmap='RdYlGn', alpha=0.5, s=25)
plt.colorbar(sc, ax=axes[1,1], label='Organic Certified')
axes[1,1].set_xlabel('Altitude (m)')
axes[1,1].set_ylabel('Quality Score')
axes[1,1].set_title('Altitude vs Quality Score', fontweight='bold')

# Quality by region
region_quality = df.groupby('region')['quality_score'].mean().sort_values(ascending=False)
axes[1,2].barh(region_quality.index, region_quality.values,
               color=sns.color_palette('husl', len(region_quality)))
axes[1,2].set_xlabel('Mean Quality Score')
axes[1,2].set_title('Mean Quality Score by Region', fontweight='bold')

plt.suptitle('Coffee Quality EDA — East Africa', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Correlations with quality_score:")
print(corr['quality_score'].drop('quality_score').abs().sort_values(ascending=False).round(3))

## 3. Preprocessing

We encode categorical variables (region, country, processing method, variety) using label encoding and split the dataset into training (80%) and test (20%) sets.

In [ ]:
le = LabelEncoder()
for col in ['region', 'country', 'processing_method', 'variety']:
    df[col + '_enc'] = le.fit_transform(df[col])

feature_cols = ['altitude_m', 'annual_rainfall_mm', 'avg_temperature_c', 'shade_percentage',
                'farm_size_ha', 'organic_certified', 'fermentation_hours', 'cherry_ripeness',
                'years_experience', 'harvest_month', 'region_enc', 'country_enc',
                'processing_method_enc', 'variety_enc']

X = df[feature_cols].values
y = df['quality_score'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Mean quality (train): {y_train.mean():.2f}, (test): {y_test.mean():.2f}")

## 4. Model Building — Random Forest

### Random Forest Algorithm

Random Forest is an **ensemble learning** method that builds multiple decision trees and combines their predictions. Key ideas:

1. **Bagging (Bootstrap Aggregation):** Each tree is trained on a bootstrap sample (random sampling with replacement) of the training data. Typically ~63% of samples appear in each tree.

2. **Feature Randomness:** At each node split, only a random subset of features is considered (typically sqrt(n_features) for classification, n_features/3 for regression). This decorrelates the trees.

3. **Aggregation:** Final prediction is the **average** of all tree predictions (regression) or majority vote (classification).

4. **Out-of-Bag (OOB) Score:** Samples not used in training a given tree can be used to estimate generalization error — a free cross-validation estimate.

**Advantages for this problem:** Handles mixed feature types, robust to outliers, captures non-linear interactions (altitude × organic × processing), naturally provides feature importance, no scaling needed.

In [ ]:
# n_estimators sweep
n_trees = [10, 25, 50, 100, 200, 300, 500]
train_rmse_list, test_rmse_list, oob_list = [], [], []

for n in n_trees:
    rf = RandomForestRegressor(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    train_pred = rf.predict(X_train)
    test_pred = rf.predict(X_test)
    train_rmse_list.append(np.sqrt(mean_squared_error(y_train, train_pred)))
    test_rmse_list.append(np.sqrt(mean_squared_error(y_test, test_pred)))
    oob_list.append(np.sqrt(mean_squared_error(y_train, rf.oob_prediction_)))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(n_trees, train_rmse_list, 'b-o', label='Train RMSE')
plt.plot(n_trees, test_rmse_list, 'r-o', label='Test RMSE')
plt.plot(n_trees, oob_list, 'g--s', label='OOB RMSE')
plt.xlabel('Number of Trees')
plt.ylabel('RMSE')
plt.title('RMSE vs Number of Trees')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# max_depth sweep
depths = [3, 5, 10, 15, 20, None]
depth_test_rmse = []
for d in depths:
    rf_d = RandomForestRegressor(n_estimators=100, max_depth=d, random_state=42, n_jobs=-1)
    rf_d.fit(X_train, y_train)
    depth_test_rmse.append(np.sqrt(mean_squared_error(y_test, rf_d.predict(X_test))))
plt.plot([str(d) for d in depths], depth_test_rmse, 'r-o', linewidth=2)
plt.xlabel('Max Depth')
plt.ylabel('Test RMSE')
plt.title('Test RMSE vs Max Depth')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
}

rf_random = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1, oob_score=True),
    param_distributions=param_dist,
    n_iter=30, cv=3, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=1
)
rf_random.fit(X_train, y_train)
print(f"Best params: {rf_random.best_params_}")
print(f"Best CV RMSE: {-rf_random.best_score_:.4f}")

best_rf = rf_random.best_estimator_
y_pred = best_rf.predict(X_test)
print(f"\nTest RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"Test R\u00b2: {r2_score(y_test, y_pred):.4f}")

## 5. Model Evaluation

We evaluate our tuned Random Forest using multiple diagnostics:
- **Actual vs Predicted scatter plot** — ideally points fall along the diagonal
- **Residual plot** — residuals should be randomly distributed around zero
- **Residuals distribution** — should be approximately normal
- **Feature importance** — both Gini (MDI) and permutation-based

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs predicted
axes[0].scatter(y_test, y_pred, alpha=0.5, s=25, color='#8B4513')
min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[0].set_xlabel('Actual Quality Score')
axes[0].set_ylabel('Predicted Quality Score')
axes[0].set_title(f'Actual vs Predicted (R\u00b2={r2_score(y_test, y_pred):.3f})', fontweight='bold')

# Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, s=25, color='steelblue')
axes[1].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Quality Score')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot', fontweight='bold')

# Residuals distribution
axes[2].hist(residuals, bins=25, edgecolor='white', color='#2ecc71', alpha=0.8)
axes[2].axvline(x=0, color='red', linestyle='--', lw=2)
axes[2].set_xlabel('Residual Value')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Residuals Distribution (std={residuals.std():.3f})', fontweight='bold')

plt.suptitle('Random Forest — Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fi_df = pd.DataFrame({
    'Feature': feature_cols,
    'Gini Importance': best_rf.feature_importances_
}).sort_values('Gini Importance', ascending=False)

# Permutation importance
perm_imp = permutation_importance(best_rf, X_test, y_test, n_repeats=10, random_state=42)
fi_df['Permutation Importance'] = perm_imp.importances_mean[
    [feature_cols.index(f) for f in fi_df['Feature']]
]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(fi_df['Feature'], fi_df['Gini Importance'],
             color=sns.color_palette('viridis', len(fi_df)))
axes[0].set_title('Gini (MDI) Feature Importance', fontweight='bold')
axes[0].set_xlabel('Mean Decrease in Impurity')

fi_perm = fi_df.sort_values('Permutation Importance', ascending=False)
axes[1].barh(fi_perm['Feature'], fi_perm['Permutation Importance'],
             color=sns.color_palette('plasma', len(fi_perm)))
axes[1].set_title('Permutation Feature Importance', fontweight='bold')
axes[1].set_xlabel('Mean Decrease in Score')
plt.tight_layout()
plt.show()
print("Top 5 important features:")
print(fi_df.head(5).to_string(index=False))

In [ ]:
cv_scores = cross_val_score(best_rf, X, y, cv=5, scoring='r2')
print(f"5-Fold CV R\u00b2: {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")

# Compare models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}
print("\nModel Comparison (5-fold CV R\u00b2):")
for name, model in models.items():
    cv = cross_val_score(model, X, y, cv=5, scoring='r2')
    print(f"  {name}: {cv.mean():.4f} \u00b1 {cv.std():.4f}")

## 6. Conclusions

### Key Findings

**Most important features for coffee quality:**
1. **Cherry ripeness** — the single strongest predictor. Harvesting fully ripe cherries is critical; under- or over-ripe cherries significantly reduce cup quality.
2. **Altitude** — higher altitude (1,800–2,200m in Ethiopia) produces slower cherry maturation, higher acidity, and more complex flavor profiles.
3. **Organic certification** — associated with better farm management practices, contributing ~3 quality points on average.
4. **Annual rainfall** — adequate and well-distributed rainfall supports cherry development.
5. **Processing method** — Natural and Honey processing add 1.5–2 quality points vs Washed, but require careful management to avoid fermentation defects.

### Practical Recommendations for Farmers

- **Focus on harvest timing**: Train pickers to select only fully red/yellow cherries; selective picking yields can increase quality scores by 5–8 points.
- **Shade management**: Maintaining 50–70% shade slows ripening and increases complexity.
- **Consider organic certification**: The premium price uplift is substantial (30–100% premium in specialty markets).
- **Processing selection**: Natural processing can achieve higher quality but requires controlled drying infrastructure.

### Model Limitations

- Synthetic data; real-world data would include soil chemistry, pest/disease history, and post-harvest logistics.
- Label encoding of regions loses geographic structure; spatial features (distance to washing station, neighboring farm quality) could improve predictions.
- The model does not account for year-to-year climate variability (ENSO effects on Ethiopian coffee growing regions).

### Next Steps

- Incorporate **soil pH, organic matter content, and trace mineral** features
- Try **Gradient Boosting (XGBoost, LightGBM)** for potentially higher accuracy
- Build **farmer-facing app** using SHAP values for explainable quality predictions
- **Time-series modeling** to predict quality trends across harvest seasons

## Appendix: Random Forest — How It Works

### The Ensemble Principle

A single decision tree suffers from **high variance** — small changes in training data can produce very different trees. Random Forest addresses this by:

$$\hat{f}_{RF}(x) = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$

where $T_b$ is the $b$-th tree trained on bootstrap sample $Z^{*b}$.

**Bias-Variance Decomposition:**
- Random Forest reduces variance while keeping bias approximately equal to a single deep tree
- The variance of the average of $B$ i.i.d. random variables each with variance $\sigma^2$ is $\sigma^2/B$
- With correlated trees (correlation $\rho$): $\text{Var} = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$
- Feature randomness reduces $\rho$, further reducing variance

### Out-of-Bag Error

Each tree uses ~63.2% of samples (since $(1-1/n)^n \to e^{-1} \approx 0.368$ samples are left out). The OOB prediction for sample $i$ uses only trees that did NOT include $i$ in training — providing an unbiased generalization estimate equivalent to leave-one-out CV.

In [ ]:
# Visualize a single tree from the forest for interpretability
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(best_rf.estimators_[0], feature_names=feature_cols, max_depth=3,
          filled=True, rounded=True, fontsize=8, ax=ax)
plt.title('Sample Tree from Random Forest (depth=3 shown)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Full tree depth: {best_rf.estimators_[0].get_depth()}")
print(f"Number of leaves: {best_rf.estimators_[0].get_n_leaves()}")

In [ ]:
# Grade-level performance analysis
df_test = df.iloc[df.index[len(X_train):]].copy() if False else pd.DataFrame(
    X_test, columns=feature_cols)
df_test['actual'] = y_test
df_test['predicted'] = y_pred
df_test['error'] = abs(y_test - y_pred)
df_test['actual_grade'] = pd.cut(y_test, bins=[60, 70, 80, 85, 90, 100],
    labels=['Commercial', 'Good', 'Very Good', 'Excellent', 'Exceptional'])

grade_perf = df_test.groupby('actual_grade')['error'].agg(['mean', 'std', 'count'])
grade_perf.columns = ['MAE', 'StdError', 'Count']
print("Model performance by quality grade:")
print(grade_perf.round(3))

grade_perf['MAE'].plot(kind='bar', color=sns.color_palette('husl', 5),
                        figsize=(9, 5), edgecolor='white')
plt.title('Mean Absolute Error by Coffee Quality Grade', fontweight='bold')
plt.xlabel('Quality Grade')
plt.ylabel('Mean Absolute Error (SCA points)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("=" * 55)
print("RANDOM FOREST — COFFEE QUALITY PREDICTION SUMMARY")
print("=" * 55)
print(f"Dataset: 800 farms from Ethiopia & Rwanda")
print(f"Features: {len(feature_cols)} agronomic/environmental variables")
print(f"Best hyperparameters: {rf_random.best_params_}")
print(f"Test RMSE:  {np.sqrt(mean_squared_error(y_test, y_pred)):.4f} SCA points")
print(f"Test MAE:   {mean_absolute_error(y_test, y_pred):.4f} SCA points")
print(f"Test R\u00b2:    {r2_score(y_test, y_pred):.4f}")
print(f"CV R\u00b2:      {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")
print(f"OOB R\u00b2:     {best_rf.oob_score_:.4f}")
print("="*55)
print(f"\nTop 3 most important features (Gini):")
for _, row in fi_df.head(3).iterrows():
    print(f"  {row['Feature']}: {row['Gini Importance']:.4f}")